# Statistical test

## Extract the .txt log of experiments

In [92]:
import re
import pandas as pd
import os

def parse_experiment_log(file_path, algorithm_name):
    # Define regex patterns for extraction
    # Matches "Iteration 1/100" or "Generation 1/100"
    iter_pattern = re.compile(r'(?:Iteration|Generation)\s+(\d+)/100')
    
    # Matches "Best Fitness: 12345" or "Current Best Fitness: 12345"
    fitness_pattern = re.compile(r'(?:Best Fitness|Current Best Fitness):\s+(\d+)')
    
    # Matches "Violations: 123"
    violation_pattern = re.compile(r'Violations:\s+(\d+)')

    data = []

    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
        # Split content into blocks by iteration headers to ensure data alignment
        # This regex splits the text while keeping the delimiter (Iteration header)
        blocks = re.split(r'(?=(?:PSO|POA|ACO|ABC)\s+(?:Iteration|Generation)\s+\d+/100)', content)
        
        for block in blocks:
            iter_match = iter_pattern.search(block)
            if iter_match:
                iteration = int(iter_match.group(1))
                
                fitness_match = fitness_pattern.search(block)
                fitness = int(fitness_match.group(1)) if fitness_match else None
                
                violation_match = violation_pattern.search(block)
                violations = int(violation_match.group(1)) if violation_match else None
                
                # Only append if we found an iteration
                data.append({
                    'iterations': iteration,
                    'fitness': fitness,
                    'violations': violations
                })

    # Create DataFrame and save to CSV
    if data:
        df = pd.DataFrame(data)
        output_filename = f"{algorithm_name.lower()}_results.csv"
        df.to_csv(output_filename, index=False)
        print(f"[Success] Extracted {len(df)} rows to {output_filename}")
    else:
        print(f"[Warning] No data found in {file_path}")

# --- Execution ---
# Map filenames to Algorithm names
path = "/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/TI/mlflow_iteration_100_ti_20/"

files_to_process = {
    path + 'pso_100.txt': 'PSO',
    path + 'poa_100.txt': 'POA',
    path + 'aco_100.txt': 'ACO',
    path + 'abc_100.txt': 'ABC'
}

# Run extraction
for file, algo in files_to_process.items():
    if os.path.exists(file):
        parse_experiment_log(file, algo)
    else:
        print(f"File not found: {file}")

[Success] Extracted 100 rows to pso_results.csv
[Success] Extracted 100 rows to poa_results.csv
[Success] Extracted 100 rows to aco_results.csv
[Success] Extracted 100 rows to abc_results.csv


## Statistical calculation

In [93]:
import pandas as pd
import numpy as np
import glob

def calculate_statistics():
    # Find all CSV files generated from the previous step
    path = "/home/emery/Documents/Scheduling_Puma_Optimizer/docs/mlflow/TI/mlflow_iteration_100_ti_20/"
    csv_files = glob.glob(path + "*_results.csv")
    
    stats_summary = []

    for file in csv_files:
        algo_name = file.replace("_results.csv", "").upper()
        df = pd.read_csv(file)
        
        # Calculate Stats for Fitness
        # In optimization, "Best" usually means Minimum Fitness (Minimization problem)
        fit_best = df['fitness'].min()
        fit_worst = df['fitness'].max()
        fit_mean = df['fitness'].mean()
        fit_std = df['fitness'].std()
        
        # Calculate Stats for Violations (handling missing data for PSO)
        if df['violations'].isnull().all():
            vio_best, vio_worst, vio_mean, vio_std = "N/A", "N/A", "N/A", "N/A"
        else:
            vio_best = df['violations'].min()
            vio_worst = df['violations'].max()
            vio_mean = df['violations'].mean()
            vio_std = df['violations'].std()

        stats_summary.append({
            'Algorithm': algo_name,
            'Fit_Best': fit_best,
            'Fit_Worst': fit_worst,
            'Fit_Mean': round(fit_mean, 2),
            'Fit_Std': round(fit_std, 2),
            'Vio_Best': vio_best,
            'Vio_Worst': vio_worst,
            'Vio_Mean': round(vio_mean, 2) if isinstance(vio_mean, (int, float)) else "N/A",
            'Vio_Std': round(vio_std, 2) if isinstance(vio_std, (int, float)) else "N/A"
        })

    # Create Summary DataFrame
    summary_df = pd.DataFrame(stats_summary)
    
    # Display the table
    print("\n--- Experiment Statistical Summary (Trajectory Analysis) ---")
    print(summary_df.to_string(index=False))
    
    # Save to CSV for reporting
    summary_df.to_csv("statistical_summary.csv", index=False)
    print("\n[Success] Summary saved to 'statistical_summary.csv'")

# --- Execution ---
calculate_statistics()


--- Experiment Statistical Summary (Trajectory Analysis) ---
                                                                                    Algorithm  Fit_Best  Fit_Worst  Fit_Mean  Fit_Std  Vio_Best  Vio_Worst  Vio_Mean  Vio_Std
/HOME/EMERY/DOCUMENTS/SCHEDULING_PUMA_OPTIMIZER/DOCS/MLFLOW/TI/MLFLOW_ITERATION_100_TI_20/ACO      1783       2354   1802.10    81.94      14.0       20.0     14.24     0.89
/HOME/EMERY/DOCUMENTS/SCHEDULING_PUMA_OPTIMIZER/DOCS/MLFLOW/TI/MLFLOW_ITERATION_100_TI_20/PSO      2072       2072   2072.00     0.00      17.0       17.0     17.00      NaN
/HOME/EMERY/DOCUMENTS/SCHEDULING_PUMA_OPTIMIZER/DOCS/MLFLOW/TI/MLFLOW_ITERATION_100_TI_20/POA       338       1887    418.77   219.38       0.0       15.0      0.71     2.09
/HOME/EMERY/DOCUMENTS/SCHEDULING_PUMA_OPTIMIZER/DOCS/MLFLOW/TI/MLFLOW_ITERATION_100_TI_20/ABC       859       1648   1142.96   212.03      13.0       36.0     21.06     6.57

[Success] Summary saved to 'statistical_summary.csv'
